# Hospital Daily Summary

Builds the gold-layer `michelin.gold.hospital_daily_summary` table from silver sources using the workbook mapping. This notebook performs a full rebuild each run and preserves rows from the driving aggregated lab results set with left joins to reference data.

In [0]:
spark.sql("SELECT current_user()").show(truncate=False)

In [0]:
catalog_name = "michelin"
target_table = f"{catalog_name}.gold.hospital_daily_summary"
source_tables = [
    (catalog_name, "silver", "lab_results"),
    (catalog_name, "silver", "hospitals"),
    (catalog_name, "silver", "dim_date"),
]


def table_is_listed(catalog_name: str, schema_name: str, table_name: str) -> bool:
    return (
        spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name} LIKE '{table_name}'").count() > 0
    )


missing_tables = [
    f"{catalog_name}.{schema_name}.{table_name}"
    for catalog_name, schema_name, table_name in source_tables
    if not table_is_listed(catalog_name, schema_name, table_name)
]

if missing_tables:
    print("Skipped build because these required source tables are not available yet:")
    for table_name in missing_tables:
        print(f" - {table_name}")
else:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")
        spark.sql(
            f"""
            CREATE OR REPLACE TABLE {target_table} AS
            WITH daily_aggregates AS (
              SELECT
                lr.hospital_id,
                CAST(lr.test_date AS DATE) AS summary_date,
                CAST(COUNT(*) AS INT) AS total_tests,
                CAST(COUNT(DISTINCT lr.patient_id) AS INT) AS unique_patients_tested,
                CAST(COUNT(DISTINCT lr.doctor_id) AS INT) AS active_doctor_count,
                CAST(SUM(CASE WHEN lr.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
                CAST(SUM(CASE WHEN lr.result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count
              FROM {catalog_name}.silver.lab_results AS lr
              GROUP BY lr.hospital_id, CAST(lr.test_date AS DATE)
            ), enriched AS (
              SELECT
                da.hospital_id,
                h.hospital_name,
                da.summary_date,
                dd.day_of_week_name AS day_name,
                dd.is_weekend,
                da.total_tests,
                da.unique_patients_tested,
                da.active_doctor_count,
                da.abnormal_test_count,
                da.critical_test_count,
                CAST(ROUND(CASE WHEN da.total_tests = 0 THEN NULL ELSE da.abnormal_test_count * 100.0 / da.total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
                CAST(ROUND(CASE WHEN da.total_tests = 0 THEN NULL ELSE da.critical_test_count * 100.0 / da.total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
              FROM daily_aggregates AS da
              LEFT JOIN {catalog_name}.silver.hospitals AS h
                ON da.hospital_id = h.hospital_id
              LEFT JOIN {catalog_name}.silver.dim_date AS dd
                ON da.summary_date = dd.calendar_date
            ), final AS (
              SELECT
                hospital_id,
                hospital_name,
                summary_date,
                day_name,
                is_weekend,
                total_tests,
                unique_patients_tested,
                active_doctor_count,
                abnormal_test_count,
                critical_test_count,
                abnormal_rate_pct,
                critical_rate_pct,
                CAST(LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS INT) AS prev_day_total_tests
              FROM enriched
            )
            SELECT
              hospital_id,
              hospital_name,
              summary_date,
              day_name,
              is_weekend,
              total_tests,
              unique_patients_tested,
              active_doctor_count,
              abnormal_test_count,
              critical_test_count,
              abnormal_rate_pct,
              critical_rate_pct,
              prev_day_total_tests,
              CAST(
                ROUND(
                  CASE
                    WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
                    ELSE (total_tests - prev_day_total_tests) * 100.0 / prev_day_total_tests
                  END,
                  2
                ) AS DECIMAL(6,2)
              ) AS test_volume_change_pct
            FROM final
            """
        )
        print(f"Created {target_table}")
    except Exception as exc:
        print(f"Build could not be completed for {target_table}.")
        print(str(exc))

In [0]:
target_table = "michelin.gold.hospital_daily_summary"

if spark.catalog.tableExists(target_table):
    preview_df = spark.table(target_table).orderBy("summary_date", "hospital_id", ascending=[False, True])
    display(preview_df.limit(50))
    print(f"Total rows: {preview_df.count():,}")
else:
    print(f"Table not created yet: {target_table}")